# GRU Leave-One-Battery-Out Cross-Validation

This notebook evaluates whether the GRU can generalize to batteries it has never seen. Each fold keeps one battery as the final test battery, uses one battery for validation, and trains on the remaining two batteries.

The scaler is fit inside each fold using only the training batteries. This prevents validation or test battery information from leaking into preprocessing.

In [ ]:
# Import the tools used for preprocessing, sequence creation, training, and evaluation.
# The notebook starts from raw data so every fold can have its own honest train-only scaler.
from pathlib import Path
import copy
import random

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

current_dir = Path.cwd().resolve()
if (current_dir / "data" / "raw" / "Battery_dataset.csv").exists():
    PROJECT_ROOT = current_dir
else:
    PROJECT_ROOT = current_dir.parent

RAW_DATA_FILE = PROJECT_ROOT / "data" / "raw" / "Battery_dataset.csv"

print("Project root:", PROJECT_ROOT)
print("Raw data file:", RAW_DATA_FILE)


In [ ]:
# Load the raw merged dataset.
# The cross-validation notebook does not reuse the current processed split because every fold needs its own scaler and windows.
raw_df = pd.read_csv(RAW_DATA_FILE)

print("Raw shape:", raw_df.shape)
print("Batteries:", sorted(raw_df["battery_id"].unique()))
raw_df.head()


In [ ]:
# Define the shared feature and target columns.
# cycle is kept for ordering only, and disT is excluded because it is constant.
FEATURE_COLUMNS = ["chI", "chV", "chT", "disI", "disV", "BCt", "SOH"]
TARGET_COLUMN = "RUL"
ID_COLUMN = "battery_id"
ORDER_COLUMN = "cycle"

WINDOW_SIZE = 10
BATCH_SIZE = 32
RANDOM_SEED = 42

INPUT_SIZE = len(FEATURE_COLUMNS)
HIDDEN_SIZE = 32
NUM_LAYERS = 1
OUTPUT_SIZE = 1
LEARNING_RATE = 0.001
MAX_EPOCHS = 500
PATIENCE = 40

model_columns = [ORDER_COLUMN, *FEATURE_COLUMNS, ID_COLUMN, TARGET_COLUMN]
model_df = raw_df.drop(columns=["disT"], errors="ignore")[model_columns].copy()

print("Input features:", FEATURE_COLUMNS)
print("Window size:", WINDOW_SIZE)
print("Model dataframe shape:", model_df.shape)


In [ ]:
# Define the folds before training.
# Each battery appears exactly once as the untouched test battery.
folds = [
    {"fold": 1, "train": ["B5", "B7"], "validation": ["B18"], "test": ["B6"]},
    {"fold": 2, "train": ["B5", "B6"], "validation": ["B18"], "test": ["B7"]},
    {"fold": 3, "train": ["B6", "B7"], "validation": ["B18"], "test": ["B5"]},
    {"fold": 4, "train": ["B5", "B6"], "validation": ["B7"], "test": ["B18"]},
]

pd.DataFrame(folds)


In [ ]:
# Scale features using training batteries only, then apply the same scaler values to validation and test.
# This repeats preprocessing inside each fold so the test battery never influences scaling.
def fit_train_min_max(train_df, feature_columns):
    feature_min = train_df[feature_columns].min()
    feature_max = train_df[feature_columns].max()
    feature_range = feature_max - feature_min
    return feature_min, feature_range


def apply_train_min_max(df, feature_columns, feature_min, feature_range):
    scaled_df = df.copy()

    for column in feature_columns:
        if feature_range[column] == 0:
            scaled_df[column] = 0.0
        else:
            scaled_df[column] = (scaled_df[column] - feature_min[column]) / feature_range[column]

    return scaled_df


In [ ]:
# Convert row-level measurements into one row per battery cycle.
# One cycle becomes one time step for the GRU.
def build_cycle_level_dataset(df):
    aggregation_rules = {column: "mean" for column in FEATURE_COLUMNS}
    aggregation_rules[TARGET_COLUMN] = "mean"

    cycle_df = (
        df[model_columns]
        .groupby([ID_COLUMN, ORDER_COLUMN], as_index=False)
        .agg(aggregation_rules)
        .sort_values([ID_COLUMN, ORDER_COLUMN])
        .reset_index(drop=True)
    )

    cycle_df[TARGET_COLUMN] = cycle_df[TARGET_COLUMN].round().astype(int)
    return cycle_df[[ORDER_COLUMN, *FEATURE_COLUMNS, ID_COLUMN, TARGET_COLUMN]]


In [ ]:
# Create sliding windows inside each battery only.
# A window contains feature history, and its label is the RUL at the last cycle of that window.
def create_sliding_windows(df, window_size):
    X_windows = []
    y_values = []
    metadata = []

    for battery_id, battery_df in df.groupby(ID_COLUMN):
        battery_df = battery_df.sort_values(ORDER_COLUMN).reset_index(drop=True)

        feature_values = battery_df[FEATURE_COLUMNS].to_numpy(dtype=np.float32)
        target_values = battery_df[TARGET_COLUMN].to_numpy(dtype=np.float32)
        cycle_values = battery_df[ORDER_COLUMN].to_numpy()

        max_start = len(battery_df) - window_size + 1
        if max_start <= 0:
            continue

        for start_idx in range(max_start):
            end_idx = start_idx + window_size
            target_idx = end_idx - 1

            X_windows.append(feature_values[start_idx:end_idx])
            y_values.append(target_values[target_idx])
            metadata.append(
                {
                    "battery_id": battery_id,
                    "start_cycle": cycle_values[start_idx],
                    "end_cycle": cycle_values[target_idx],
                    "target_RUL": target_values[target_idx],
                }
            )

    X = np.array(X_windows, dtype=np.float32)
    y = np.array(y_values, dtype=np.float32)
    window_metadata = pd.DataFrame(metadata)
    return X, y, window_metadata


In [ ]:
# Define the same GRU architecture used in the first experiment.
# Keeping architecture fixed makes the fold results comparable.
class GRURULModel(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, output_size):
        super().__init__()

        self.gru = nn.GRU(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
        )

        self.fc = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        gru_output, hidden = self.gru(x)
        final_hidden_state = hidden[-1]
        prediction = self.fc(final_hidden_state)
        return prediction


In [ ]:
# Build all datasets and loaders for one fold.
# This function is the core guard against leakage because scaling is fit only on fold training data.
def prepare_fold_data(fold):
    train_raw = model_df[model_df[ID_COLUMN].isin(fold["train"])].copy()
    validation_raw = model_df[model_df[ID_COLUMN].isin(fold["validation"])].copy()
    test_raw = model_df[model_df[ID_COLUMN].isin(fold["test"])].copy()

    split_sets = {
        "train": set(train_raw[ID_COLUMN].unique()),
        "validation": set(validation_raw[ID_COLUMN].unique()),
        "test": set(test_raw[ID_COLUMN].unique()),
    }

    split_names = list(split_sets)
    for left_index, left_name in enumerate(split_names):
        for right_name in split_names[left_index + 1:]:
            overlap = split_sets[left_name] & split_sets[right_name]
            if overlap:
                raise ValueError(f"{left_name}/{right_name} overlap found: {sorted(overlap)}")

    feature_min, feature_range = fit_train_min_max(train_raw, FEATURE_COLUMNS)

    train_scaled = apply_train_min_max(train_raw, FEATURE_COLUMNS, feature_min, feature_range)
    validation_scaled = apply_train_min_max(validation_raw, FEATURE_COLUMNS, feature_min, feature_range)
    test_scaled = apply_train_min_max(test_raw, FEATURE_COLUMNS, feature_min, feature_range)

    train_cycle_df = build_cycle_level_dataset(train_scaled)
    validation_cycle_df = build_cycle_level_dataset(validation_scaled)
    test_cycle_df = build_cycle_level_dataset(test_scaled)

    X_train, y_train, train_metadata = create_sliding_windows(train_cycle_df, WINDOW_SIZE)
    X_val, y_val, validation_metadata = create_sliding_windows(validation_cycle_df, WINDOW_SIZE)
    X_test, y_test, test_metadata = create_sliding_windows(test_cycle_df, WINDOW_SIZE)

    X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
    y_train_tensor = torch.tensor(y_train, dtype=torch.float32).view(-1, 1)
    X_val_tensor = torch.tensor(X_val, dtype=torch.float32)
    y_val_tensor = torch.tensor(y_val, dtype=torch.float32).view(-1, 1)
    X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
    y_test_tensor = torch.tensor(y_test, dtype=torch.float32).view(-1, 1)

    train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
    val_dataset = TensorDataset(X_val_tensor, y_val_tensor)
    test_dataset = TensorDataset(X_test_tensor, y_test_tensor)

    return {
        "train_dataset": train_dataset,
        "val_dataset": val_dataset,
        "test_dataset": test_dataset,
        "test_metadata": test_metadata,
        "shapes": {
            "X_train": X_train.shape,
            "X_val": X_val.shape,
            "X_test": X_test.shape,
        },
    }


In [ ]:
# Train one fold and return its test metrics.
# The same seed is offset by fold number so each fold is reproducible but independent.
def train_and_evaluate_fold(fold):
    seed = RANDOM_SEED + fold["fold"]
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    fold_data = prepare_fold_data(fold)

    train_generator = torch.Generator()
    train_generator.manual_seed(seed)

    train_loader = DataLoader(
        fold_data["train_dataset"],
        batch_size=BATCH_SIZE,
        shuffle=True,
        generator=train_generator,
    )
    val_loader = DataLoader(fold_data["val_dataset"], batch_size=BATCH_SIZE, shuffle=False)
    test_loader = DataLoader(fold_data["test_dataset"], batch_size=BATCH_SIZE, shuffle=False)

    model = GRURULModel(INPUT_SIZE, HIDDEN_SIZE, NUM_LAYERS, OUTPUT_SIZE)
    loss_function = nn.MSELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

    train_losses = []
    val_losses = []
    best_val_loss = float("inf")
    best_epoch = 0
    best_model_state = None
    epochs_without_improvement = 0

    for epoch in range(MAX_EPOCHS):
        model.train()
        total_train_loss = 0.0

        for batch_X, batch_y in train_loader:
            optimizer.zero_grad()
            predictions = model(batch_X)
            loss = loss_function(predictions, batch_y)
            loss.backward()
            optimizer.step()
            total_train_loss += loss.item() * batch_X.size(0)

        average_train_loss = total_train_loss / len(train_loader.dataset)
        train_losses.append(average_train_loss)

        model.eval()
        total_val_loss = 0.0

        with torch.no_grad():
            for batch_X, batch_y in val_loader:
                predictions = model(batch_X)
                loss = loss_function(predictions, batch_y)
                total_val_loss += loss.item() * batch_X.size(0)

        average_val_loss = total_val_loss / len(val_loader.dataset)
        val_losses.append(average_val_loss)

        if average_val_loss < best_val_loss:
            best_val_loss = average_val_loss
            best_epoch = epoch + 1
            best_model_state = copy.deepcopy(model.state_dict())
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1

        if epochs_without_improvement >= PATIENCE:
            break

    model.load_state_dict(best_model_state)
    model.eval()

    test_predictions = []
    test_targets = []

    with torch.no_grad():
        for batch_X, batch_y in test_loader:
            predictions = model(batch_X)
            test_predictions.append(predictions)
            test_targets.append(batch_y)

    test_predictions = torch.cat(test_predictions).squeeze().numpy()
    test_targets = torch.cat(test_targets).squeeze().numpy()

    test_mae = np.mean(np.abs(test_predictions - test_targets))
    test_rmse = np.sqrt(np.mean((test_predictions - test_targets) ** 2))

    return {
        "fold": fold["fold"],
        "train_batteries": ", ".join(fold["train"]),
        "validation_battery": ", ".join(fold["validation"]),
        "test_battery": ", ".join(fold["test"]),
        "best_epoch": best_epoch,
        "best_val_rmse": np.sqrt(best_val_loss),
        "test_mae": test_mae,
        "test_rmse": test_rmse,
        "train_windows": fold_data["shapes"]["X_train"][0],
        "validation_windows": fold_data["shapes"]["X_val"][0],
        "test_windows": fold_data["shapes"]["X_test"][0],
        "train_losses": train_losses,
        "val_losses": val_losses,
        "test_predictions": test_predictions,
        "test_targets": test_targets,
    }


In [ ]:
# Run all cross-validation folds.
# This may take several minutes because each fold trains a fresh GRU model.
fold_results = []

for fold in folds:
    print(f"Running fold {fold['fold']} | test battery: {fold['test']}")
    result = train_and_evaluate_fold(fold)
    fold_results.append(result)
    print(
        f"Fold {result['fold']} complete | "
        f"Best Val RMSE: {result['best_val_rmse']:.2f} | "
        f"Test MAE: {result['test_mae']:.2f} | "
        f"Test RMSE: {result['test_rmse']:.2f}"
    )


In [ ]:
# Summarize fold results in one table.
# The average test RMSE is a better project-level estimate than one lucky or unlucky split.
results_df = pd.DataFrame(
    [
        {
            "fold": result["fold"],
            "train_batteries": result["train_batteries"],
            "validation_battery": result["validation_battery"],
            "test_battery": result["test_battery"],
            "best_epoch": result["best_epoch"],
            "best_val_rmse": result["best_val_rmse"],
            "test_mae": result["test_mae"],
            "test_rmse": result["test_rmse"],
            "train_windows": result["train_windows"],
            "validation_windows": result["validation_windows"],
            "test_windows": result["test_windows"],
        }
        for result in fold_results
    ]
)

results_df


In [ ]:
# Report average performance across folds.
# This is the main cross-validation summary for the GRU model.
summary_metrics = results_df[["test_mae", "test_rmse"]].agg(["mean", "std"])
summary_metrics


In [ ]:
# Plot test RMSE by held-out battery to see which battery is hardest for the GRU.
plt.figure(figsize=(8, 5))
plt.bar(results_df["test_battery"], results_df["test_rmse"])
plt.xlabel("Held-out test battery")
plt.ylabel("Test RMSE (cycles)")
plt.title("GRU Leave-One-Battery-Out Test RMSE")
plt.grid(axis="y")
plt.show()
